# SQL

In [12]:
SQLITE_PATH = './data/dbfinal.sqlite3'
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

def execute_query(db, query):
    conn = sqlite3.connect(db)
    cur = conn.cursor()
    cur.execute(query)
    rows = cur.fetchall()
    conn.commit()
    conn.close()
    return rows

In [3]:
execute_query(SQLITE_PATH, "DELETE FROM moves WHERE white_elo < 1700 OR white_elo >= 1800;")
execute_query(SQLITE_PATH, "DELETE FROM moves WHERE type = 'Bullet' OR type = 'UltraBullet' OR type = 'Correspondence';")
execute_query(SQLITE_PATH, "DELETE FROM moves WHERE white_active = 0;")
execute_query(SQLITE_PATH, "DELETE FROM moves WHERE low_time = 1;")

[]

In [7]:
execute_query(SQLITE_PATH, "VACUUM;")

[]

In [8]:
drop_columns = ['result', 'white_player', 'black_player', 'black_elo', 'time_control', 'termination', 'white_won', 'black_won', 'no_winner', 'num_ply', 'winrate', 'winrate_elo', 'winrate_loss', 'is_blunder_wr', 'opp_winrate', 'active_elo', 'opponent_elo', 'active_won', 'is_capture', 'clock', 'opp_clock', 'clock_percent', 'opp_clock_percent', 'active_bishop_count', 'active_knight_count','active_pawn_count','active_queen_count','active_rook_count', 'opp_bishop_count', 'opp_knight_count','opp_pawn_count','opp_queen_count','opp_rook_count', 'type', 'white_elo', 'move_ply', 'cp', 'cp_loss', 'cp_rel', 'white_active', 'low_time', 'num_legal_moves']

In [9]:
for column in drop_columns:
    execute_query(SQLITE_PATH, "ALTER TABLE moves DROP COLUMN {};".format(column))
execute_query(SQLITE_PATH, "VACUUM;")

[]

In [16]:
execute_query(SQLITE_PATH, "PRAGMA table_info(moves);")

[(0, 'move', 'TEXT', 0, None, 0),
 (1, 'is_blunder_cp', 'INTEGER', 0, None, 0),
 (2, 'board', 'TEXT', 0, None, 0),
 (3, 'is_check', 'INTEGER', 0, None, 0)]

In [11]:
execute_query(SQLITE_PATH, "SELECT COUNT(*) FROM moves;")

[(4828128,)]

# Create Board Data - One Hot

In [4]:
data = pd.DataFrame(execute_query(SQLITE_PATH, "SELECT * FROM moves;"))
data.rename(columns={0:'move', 1:'is_blunder_cp', 2:'board', 3:'is_check'}, inplace=True)

In [5]:
data

,move,is_blunder_cp,board,is_check
0,e2e4,0,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,0
1,g1f3,0,rnbqkbnr/pppp1ppp/4p3/8/4P3/8/PPPP1PPP/RNBQKBN...,0
2,d2d4,0,rnbqkbnr/1ppp1ppp/p3p3/8/4P3/5N2/PPPP1PPP/RNBQ...,0
3,b2b3,0,rnbqkbnr/2pp1ppp/p3p3/1p6/3PP3/5N2/PPP2PPP/RNB...,0
4,f1d3,0,rn1qkbnr/1bpp1ppp/p3p3/1p6/3PP3/1P3N2/P1P2PPP/...,0
...,...,...,...,...
4828123,h7h8,0,2r2k2/p4ppQ/2p2b1p/3p4/q2P2Pn/P2BP2P/5P2/R1R3K...,0
4828124,h8c8,0,2r4Q/p3kpp1/2p2b1p/3p4/q2P2Pn/P2BP2P/5P2/R1R3K...,0
4828125,g1g2,0,2Q5/p3kpp1/2p2b1p/3p4/q2P2P1/P2BPn1P/5P2/R1R3K...,1
4828126,g2g3,1,2Q5/p3kpp1/2p2b1p/3p4/q2P2Pn/P2BP2P/5PK1/R1R5 ...,1


## Add castle rights

In [6]:
def get_castle_from_board(boards):
    list_splitted = [xi.split(sep=' ') for xi in boards]
    return [row[2] for row in list_splitted]

def get_rights_column(right, data):
    rights = []
    for r in data:
        if right in r:
            rights.append(1)
        else:
            rights.append(0)
    return rights

def add_castle_rights(data):
    rights = get_castle_from_board(data['board'])
    data['white_king_rights'] = get_rights_column('K', rights)
    data['white_queen_rights'] = get_rights_column('Q', rights)
    data['black_king_rights'] = get_rights_column('k', rights)
    data['black_queen_rights'] = get_rights_column('q', rights)
    return data

In [7]:
data = add_castle_rights(data)
data

,move,is_blunder_cp,board,is_check,white_king_rights,white_queen_rights,black_king_rights,black_queen_rights
0,e2e4,0,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,0,1,1,1,1
1,g1f3,0,rnbqkbnr/pppp1ppp/4p3/8/4P3/8/PPPP1PPP/RNBQKBN...,0,1,1,1,1
2,d2d4,0,rnbqkbnr/1ppp1ppp/p3p3/8/4P3/5N2/PPPP1PPP/RNBQ...,0,1,1,1,1
3,b2b3,0,rnbqkbnr/2pp1ppp/p3p3/1p6/3PP3/5N2/PPP2PPP/RNB...,0,1,1,1,1
4,f1d3,0,rn1qkbnr/1bpp1ppp/p3p3/1p6/3PP3/1P3N2/P1P2PPP/...,0,1,1,1,1
...,...,...,...,...,...,...,...,...
4828123,h7h8,0,2r2k2/p4ppQ/2p2b1p/3p4/q2P2Pn/P2BP2P/5P2/R1R3K...,0,0,0,0,0
4828124,h8c8,0,2r4Q/p3kpp1/2p2b1p/3p4/q2P2Pn/P2BP2P/5P2/R1R3K...,0,0,0,0,0
4828125,g1g2,0,2Q5/p3kpp1/2p2b1p/3p4/q2P2P1/P2BPn1P/5P2/R1R3K...,1,0,0,0,0
4828126,g2g3,1,2Q5/p3kpp1/2p2b1p/3p4/q2P2Pn/P2BP2P/5PK1/R1R5 ...,1,0,0,0,0


## Board to One-hot

In [8]:
piece_to_one_hot = {
    'r':[-1.,0.,0.,0.,0.,0.],
    'n':[0.,-1.,0.,0.,0.,0.],
    'b':[0.,0.,-1.,0.,0.,0.],
    'q':[0.,0.,0.,-1.,0.,0.],
    'k':[0.,0.,0.,0.,-1.,0.],
    'p':[0.,0.,0.,0.,0.,-1.],
    'R':[1.,0.,0.,0.,0.,0.],
    'N':[0.,1.,0.,0.,0.,0.],
    'B':[0.,0.,1.,0.,0.,0.],
    'Q':[0.,0.,0.,1.,0.,0.],
    'K':[0.,0.,0.,0.,1.,0.],
    'P':[0.,0.,0.,0.,0.,1.]
}
empty_square = [0.,0.,0.,0.,0.,0.]

def leave_board_only(data):
    data['board'] = [board.split(sep=' ')[0] for board in data['board']]
    return data

def get_one_hot(board: str):
    one_hot_board = []
    rows = board.split(sep='/')
    for row in rows:
        one_hot_row = []
        for piece in [*row]:
            if piece.isdigit():
                for i in range(int(piece)):
                    one_hot_row.append(empty_square)
            else:
                one_hot_row.append(piece_to_one_hot[piece])
        one_hot_board.append(one_hot_row)
    return one_hot_board


def boards_to_one_hot(data):
    data = leave_board_only(data)
    data['board'] = data['board'].apply(get_one_hot)

In [9]:
boards_to_one_hot(data)
data

,move,is_blunder_cp,board,is_check,white_king_rights,white_queen_rights,black_king_rights,black_queen_rights
0,e2e4,0,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, -1.0,...",0,1,1,1,1
1,g1f3,0,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, -1.0,...",0,1,1,1,1
2,d2d4,0,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, -1.0,...",0,1,1,1,1
3,b2b3,0,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, -1.0,...",0,1,1,1,1
4,f1d3,0,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, -1.0,...",0,1,1,1,1
...,...,...,...,...,...,...,...,...
4828123,h7h8,0,"[[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0...",0,0,0,0,0
4828124,h8c8,0,"[[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0...",0,0,0,0,0
4828125,g1g2,0,"[[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0...",1,0,0,0,0
4828126,g2g3,1,"[[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0...",1,0,0,0,0


## Separar movimiento en posicion inicial y final

In [10]:
data['initial_pos'] = [move[:2] for move in data['move']]
data['final_pos'] = [move[2:4] for move in data['move']]

In [11]:
data = data[['board', 'white_king_rights', 'white_queen_rights', 'black_king_rights', 'black_queen_rights', 'initial_pos', 'final_pos', 'is_blunder_cp', 'is_check']]
data

,board,white_king_rights,white_queen_rights,black_king_rights,black_queen_rights,initial_pos,final_pos,is_blunder_cp,is_check
0,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, -1.0,...",1,1,1,1,e2,e4,0,0
1,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, -1.0,...",1,1,1,1,g1,f3,0,0
2,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, -1.0,...",1,1,1,1,d2,d4,0,0
3,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, -1.0,...",1,1,1,1,b2,b3,0,0
4,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, -1.0,...",1,1,1,1,f1,d3,0,0
...,...,...,...,...,...,...,...,...,...
4828123,"[[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0...",0,0,0,0,h7,h8,0,0
4828124,"[[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0...",0,0,0,0,h8,c8,0,0
4828125,"[[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0...",0,0,0,0,g1,g2,0,1
4828126,"[[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0...",0,0,0,0,g2,g3,1,1


In [17]:
train_val_df, test_df = train_test_split(data, test_size=0.1, random_state=47, stratify=data['initial_pos'])
train_df, val_df = train_test_split(train_val_df, test_size=0.1, random_state=47, stratify=train_val_df['initial_pos'])

In [18]:
print("Train: {}".format(len(train_df)))
print("Validation: {}".format(len(val_df)))
print("Test: {}".format(len(test_df)))

Train: 3910783
Validation: 434532
Test: 482813


## Almacenar en un csv

In [19]:
train_df.to_csv("./data/final_data/ml_final_data_train.csv")
val_df.to_csv("./data/final_data/ml_final_data_val.csv")
test_df.to_csv("./data/final_data/ml_final_data_test.csv")

### Almacenar en BD

In [24]:
# SQLITE_ML_PATH = './data/db_ml_final.sqlite3'

# def serialize_board(board):
#     return np.array(board).tobytes()

# def create_final_db_ml(data):
#     conn = sqlite3.connect(SQLITE_ML_PATH)
#     cursor = conn.cursor()
#     cursor.execute("""
#     CREATE TABLE IF NOT EXISTS moves (
#         board BLOB,
#         white_king_rights INTEGER,
#         white_queen_rights INTEGER,
#         black_king_rights INTEGER,
#         black_queen_rights INTEGER,
#         initial_pos TEXT,
#         final_pos TEXT,
#         is_blunder_cp INTEGER,
#         is_check INTEGER
#     )
#     """)
#     conn.commit()

#     for index, row in data.iterrows():
#         board_blob = serialize_board(row['board'])
#         cursor.execute("INSERT INTO moves (board, white_king_rights, white_queen_rights, black_king_rights, black_queen_rights, initial_pos, final_pos, is_blunder_cp, is_check) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)", (board_blob, row['white_king_rights'], row['white_queen_rights'], row['black_king_rights'], row['black_queen_rights'], row['initial_pos'], row['final_pos'], row['is_blunder_cp'], row['is_check']))
    
#     conn.commit()
#     conn.close()


In [25]:
# create_final_db_ml(data)